In [ ]:
!pip install tensorflow

In [5]:
import tensorflow as tf

In [6]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras import Input


In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
# Load the pre-trained model (ResNet50 or VGG16)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [10]:
# Freeze the base model layers to retain pre-trained weights
for layer in base_model.layers:
    layer.trainable = False

In [11]:

# Add custom layers for feature extraction
x = base_model.output
x = Flatten()(x)  # Flatten the output of the convolutional layers
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

In [ ]:
num_clothing_types = 

In [ ]:
num_colors

In [ ]:

# Output 1: Clothing Type
clothing_type_output = Dense(num_clothing_types, activation='softmax', name='clothing_type_output')(x)

# Output 2: Color
color_output = Dense(num_colors, activation='softmax', name='color_output')(x)

# Output 3: Shade
shade_output = Dense(num_shades, activation='softmax', name='shade_output')(x)

# Output 4: Texture
texture_output = Dense(num_textures, activation='softmax', name='texture_output')(x)

# Output 5: Material Type
material_output = Dense(num_materials, activation='softmax', name='material_output')(x)

# Output 6: Product Name (optional, if extracting from image text/logos)
product_output = Dense(num_product_names, activation='softmax', name='product_output')(x)

# Create the multi-output model
model = Model(inputs=base_model.input, outputs=[clothing_type_output, color_output, shade_output, texture_output, material_output, product_output])


In [14]:
from tensorflow.keras.utils import load_img, img_to_array

In [ ]:
# Load a new image for prediction
image = load_img(r"C:\Users\pavan\Downloads\zalando-hd-resized\test\cloth\14673_00.jpg", target_size=(224, 224))
image = img_to_array(image)
image = np.expand_dims(image, axis=0)

# Predict clothing features
predictions = base_model.predict(image)
clothing_type_pred = predictions[0]
color_pred = predictions[1]
shade_pred = predictions[2]
texture_pred = predictions[3]
material_pred = predictions[4]
product_name_pred = predictions[5]


In [ ]:
predictions.shape

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

# Load the pre-trained ResNet50 model without the top classification layers
model = ResNet50(weights='imagenet', include_top=False)  # Set include_top=False to remove the final FC layers

# The model will now output feature maps instead of class probabilities.
model.summary()


In [19]:
# Load and preprocess the image
image_path = r"C:\Users\pavan\Downloads\zalando-hd-resized\test\cloth\14673_00.jpg"
img = load_img(image_path, target_size=(224, 224))  # Resize the image
img_array = img_to_array(img)

# Expand the dimensions to match the model's input (batch, height, width, channels)
img_array = np.expand_dims(img_array, axis=0)

# Preprocess the image for the specific model (ResNet50 in this case)
img_array = preprocess_input(img_array)  # Model-specific preprocessing (ImageNet-trained models)


In [ ]:
# Extract features from the image using the pre-trained model
features = model.predict(img_array)

# The extracted features will be a multi-dimensional array (e.g., (1, 7, 7, 2048) for ResNet50)
print('Feature shape:', features.shape)

# You can reshape it into a vector if needed for further processing (e.g., for use in a classifier)
features_flattened = features.reshape(features.shape[0], -1)
print('Flattened feature shape:', features_flattened.shape)


In [21]:
# Example: Save the features to a file
np.save('image_features.npy', features_flattened)

# You can later load this using np.load('image_features.npy')


In [ ]:
from tensorflow.keras.applications import VGG16

# Load the pre-trained VGG16 model without the classification layers
model_vgg = VGG16(weights='imagenet', include_top=False)


In [ ]:
# Define a new model that outputs features from an intermediate layer (e.g., 'conv5_block3_out' in ResNet50)
from tensorflow.keras.models import Model

layer_name = 'conv5_block3_out'  # Example layer name in ResNet50
intermediate_layer_model = Model(inputs=model.input, outputs=model.get_layer(layer_name).output)

# Get features from this specific layer
intermediate_features = intermediate_layer_model.predict(img_array)
print('Intermediate feature shape:', intermediate_features.shape)


In [ ]:
intermediate_features

In [ ]:
%pip install webcolors

In [28]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from sklearn.cluster import KMeans
from collections import Counter


In [ ]:
# Load the pre-trained MobileNetV2 model
model = tf.keras.applications.MobileNetV2(weights='imagenet')

# Function to preprocess the image
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))  # Resize to 224x224
    img = image.img_to_array(img)       # Convert to array
    img = np.expand_dims(img, axis=0)   # Add batch dimension
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)  # Preprocess
    return img


In [30]:
def classify_clothing(img_path):
    img = preprocess_image(img_path)
    preds = model.predict(img)  # Make predictions
    decoded_preds = tf.keras.applications.mobilenet_v2.decode_predictions(preds, top=3)[0]
    clothing_type = decoded_preds[0][1]  # Get the highest predicted class
    return clothing_type


In [31]:
def extract_dominant_color(img_path, num_colors=5):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.reshape((img.shape[0] * img.shape[1], 3))  # Reshape to a 2D array of pixels

    kmeans = KMeans(n_clusters=num_colors)
    kmeans.fit(img)

    # Get the most common colors
    colors = Counter(kmeans.labels_)
    most_common_colors = colors.most_common(1)
    return kmeans.cluster_centers_[most_common_colors[0][0]].astype(int)


In [36]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def apply_gabor_filter(image):
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Define Gabor parameters
    kernels = []
    for theta in range(4):  # Different orientations
        theta = theta / 4. * np.pi
        for sigma in (1, 3):  # Different scales
            for lamda in (np.pi / 4, np.pi / 2):  # Different wavelengths
                gabor_kernel = cv2.getGaborKernel((21, 21), sigma, theta, lamda, 0.5, 0, ktype=cv2.CV_32F)
                kernels.append(gabor_kernel)

    # Apply each Gabor filter to the image
    filtered_images = []
    for kernel in kernels:
        filtered_image = cv2.filter2D(gray_image, cv2.CV_8UC3, kernel)
        filtered_images.append(filtered_image)

    return filtered_images

def show_filtered_images(images):
    for idx, img in enumerate(images):
        plt.subplot(2, 4, idx + 1)
        plt.imshow(img, cmap='gray')
        plt.axis('off')
    plt.show()


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

# Prepare your data (assuming you have a dataset in a directory)
datagen = ImageDataGenerator(rescale=1.0/255.0)
train_generator = datagen.flow_from_directory(
    r"C:\Users\pavan\Downloads\zalando-hd-resized\train\cloth",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Build the model
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(layers.MaxPooling2D((2, 2)))
# Add more layers as needed...

# Compile and train
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_generator, epochs=10)


In [38]:
def analyze_clothing_image(img_path):
    clothing_type = classify_clothing(img_path)
    dominant_color = extract_dominant_color(img_path)

    # Get the actual color name from the RGB values
    actual_color_name = rgb_to_color_name(dominant_color)

    # Use the RGB values for detailed description
    color_rgb = f"RGB: {dominant_color[0]}, {dominant_color[1]}, {dominant_color[2]}"

    # Note: Texture and material extraction requires more complex models or datasets.
    texture = "Smooth"  # Placeholder
    material = "Cotton"  # Placeholder

    # Detailed description (you might refine this based on actual attributes)
    detailed_description = f"A {clothing_type} in {actual_color_name} ({color_rgb}) with a {texture} texture, made from {material}."

    return {
        "product_name": "Stylish Dress",  # Placeholder
        "clothing_type": clothing_type,
        "color": actual_color_name,
        "shade": color_rgb,
        "texture": texture,
        "material_type": material,
        "detailed_description": detailed_description
    }


In [35]:
import webcolors

def rgb_to_color_name(rgb):
    try:
        # Try to get the exact color name
        color_name = webcolors.rgb_to_name(rgb)
    except ValueError:
        # If there's no exact name, find the closest one
        closest_name = min(webcolors.CSS3_HEX_TO_NAMES.keys(), key=lambda name: webcolors.hex_to_rgb(webcolors.CSS3_HEX_TO_NAMES[name]))
        closest_rgb = webcolors.hex_to_rgb(closest_name)
        color_name = webcolors.rgb_to_name(closest_rgb)
    return color_name


In [ ]:
img_path = r"C:\Users\pavan\Downloads\zalando-hd-resized\test\cloth\14673_00.jpg"  # Change this to the path of your clothing image
result = analyze_clothing_image(img_path)
print(result)


In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
#from wordcloud import WordCloud, STOPWORDS
from datetime import datetime
from PIL import Image

In [ ]:
import torch

print("Number of GPU: ", torch.cuda.device_count())
print("GPU Name: ", torch.cuda.get_device_name())

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
total_folders = total_files = 0
folder_info = []
images_names = []
for base, dirs, files in tqdm(os.walk(r"C:\Users\pavan\Downloads\images")):
    for directories in dirs:
        folder_info.append((directories, len(os.listdir(os.path.join(base, directories)))))
        total_folders += 1
    for _files in files:
        total_files += 1
        if len(_files.split(".jpg"))==2:
            images_names.append(_files.split(".jpg")[0])

In [5]:
image_name_df = pd.DataFrame(images_names, columns = ["image_name"])
image_name_df["article_id"] = image_name_df["image_name"].apply(lambda x: int(x[1:]))

In [ ]:
print(f"Total number of folders: {total_folders}\nTotal number of files: {total_files}")
folder_info_df = pd.DataFrame(folder_info, columns=["folder", "files count"])
folder_info_df.sort_values(["files count"], ascending=False).head()

In [ ]:
print("folder names: ", list(folder_info_df.folder.unique()))

In [8]:
articles_df = pd.read_csv(r"C:\Users\pavan\Downloads\articles.csv\articles.csv")

In [ ]:
image_article_df = articles_df[["article_id", "product_code", "product_group_name", "product_type_name"]].merge(image_name_df, on=["article_id"], how="left")
print(image_article_df.shape)
image_article_df.head()

In [ ]:
article_no_image_df = image_article_df.loc[image_article_df.image_name.isna()]
print(article_no_image_df.shape)
article_no_image_df.head()

In [ ]:
print("Product codes with some missing images: ", article_no_image_df.product_code.nunique())
print("Product groups with some missing images: ", list(article_no_image_df.product_group_name.unique()))

In [12]:
def plot_image_samples(image_article_df, product_group_name, cols=1, rows=-1):
    image_path = r"C:\Users\pavan\Downloads\images" 
    _df = image_article_df.loc[image_article_df.product_group_name==product_group_name]
    article_ids = _df.article_id.values[0:cols*rows]
    plt.figure(figsize=(2 + 3 * cols, 2 + 4 * rows))
    for i in range(cols * rows):
        article_id = ("0" + str(article_ids[i]))[-10:]
        plt.subplot(rows, cols, i + 1)
        plt.axis('off')
        plt.title(f"{product_group_name} {article_id[:3]}\n{article_id}.jpg")
        image = Image.open(f"{image_path}/{article_id[:3]}/{article_id}.jpg")
        plt.imshow(image)

In [ ]:
print(image_article_df.product_group_name.unique())

In [ ]:
plot_image_samples(image_article_df, "Garment Lower body", 4, 2)

In [ ]:
plot_image_samples(image_article_df, "Stationery", 4, 1)

In [ ]:
plot_image_samples(image_article_df, "Fun", 2, 1)

In [ ]:
plot_image_samples(image_article_df, "Accessories", 4, 1)

In [ ]:
plot_image_samples(image_article_df, "Swimwear", 4, 2)

In [ ]:
plot_image_samples(image_article_df, "Bags", 4, 3)

In [20]:
import torch
import torchvision.models as models
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import numpy as np


In [21]:
import torch
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np


In [24]:
# Define the directory with all images
image_dir = r"C:\Users\pavan\Downloads\images\094"  # Your single image folder path

# List all image files in the directory
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]


In [25]:

# Step 2: Define image transformations (Resizing and Normalization)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [26]:

# Function to load and preprocess images
def load_and_transform_image(image_path):
    image = Image.open(image_path).convert('RGB')  # Convert to RGB to ensure 3 channels
    return data_transforms(image).unsqueeze(0)  # Add batch dimension


In [27]:

# Load and preprocess all images
image_tensors = []
for img_file in image_files:
    img_path = os.path.join(image_dir, img_file)
    img_tensor = load_and_transform_image(img_path)
    image_tensors.append(img_tensor)


In [ ]:

# Stack all images into a single tensor (batch size x channels x height x width)
image_tensors = torch.cat(image_tensors)

print(f"Loaded {len(image_tensors)} images with shape: {image_tensors.shape}")



In [29]:
import torchvision.models as models
import torch


In [ ]:

# Step 3: Load the pre-trained ResNet50 model
model = models.resnet50(pretrained=True)
model.fc = torch.nn.Identity()  # Remove the final classification layer to get feature vectors
model.eval()  # Set model to evaluation mode


In [31]:

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [32]:

# Step 4: Extract features
image_tensors = image_tensors.to(device)  # Move images to GPU if available


In [ ]:

# Split into smaller batches if memory is limited
batch_size = 32
features = []
for i in range(0, len(image_tensors), batch_size):
    batch = image_tensors[i:i+batch_size]
    with torch.no_grad():
        outputs = model(batch)  # Get feature vectors
    features.append(outputs.cpu().numpy())  # Move to CPU and convert to numpy

# Concatenate all feature vectors
features = np.vstack(features)
print("Feature extraction complete. Feature matrix shape:", features.shape)



In [ ]:
from sklearn.decomposition import PCA

# Step 5: Perform PCA
pca = PCA(n_components=50)  # Reduce to 50 components
features_pca = pca.fit_transform(features)

print("PCA Completed: Shape", features_pca.shape)


In [ ]:
from sklearn.cluster import KMeans

# Step 6: Perform K-Means Clustering
kmeans = KMeans(n_clusters=4, random_state=42)  # Assuming 4 seasons (spring, summer, fall, winter)
cluster_labels = kmeans.fit_predict(features_pca)

print("K-Means clustering completed.")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Function to show random images from a specific cluster
def show_images_from_cluster(cluster_num, num_images=5):
    cluster_indices = np.where(cluster_labels == cluster_num)[0]
    selected_indices = np.random.choice(cluster_indices, size=num_images, replace=False)
    
    for idx in selected_indices:
        img_file = image_files[idx]
        img_path = os.path.join(image_dir, img_file)
        img = Image.open(img_path)
        plt.imshow(img)
        plt.title(f"Cluster {cluster_num}")
        plt.show()

# Example: Show 5 random images from Cluster 0
show_images_from_cluster(0, num_images=5)


In [ ]:
show_images_from_cluster(3, num_images=10)

In [ ]:
# Manually map clusters to seasons
cluster_to_season = {
    0: 'Winter',
    1: 'Summer',
    2: 'Spring',
    3: 'Fall'
}

# Assign season labels to the cluster predictions
season_predictions = [cluster_to_season[label] for label in cluster_labels]

# Print the predicted season for each image
for img_file, season in zip(image_files, season_predictions):
    print(f"Image: {img_file} --> Season: {season}")


In [ ]:

import pandas as pd

# Save the results
results_df = pd.DataFrame({
    'Image_File': image_files,
    'Predicted_Season': season_predictions
})

results_df.to_csv(r"C:\Users\pavan\Downloads\seasons.csv", index=False)
print("Predictions saved to 'season_predictions.csv'")


In [ ]:
results_df

In [ ]:
image_article_df[image_article_df['image_name']=='0949594001']

In [103]:
winter_l = results_df[results_df['Predicted_Season']=='Winter']['Image_File'].to_list()

In [ ]:
image = Image.open(r"C:\Users\pavan\Downloads\images\094\0949551002.jpg")
plt.imshow(image)